# QuantumHive — Entrenamiento Completo en Kaggle GPU
**Dataset:** `quantumhive_fusion` (tus CSVs de MT5)\n**Accelerator:** GPU T4

In [ ]:
# Celda 0 — Instalar dependencias (correr primero)
!pip install -q stable-baselines3 sb3-contrib gymnasium onnx onnxruntime pandas numpy pyarrow
print('Dependencias listas')

In [ ]:
# Celda 1 — Imports y verificacion GPU
from __future__ import annotations
import os, sys, json, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
import gymnasium as gym
from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import DummyVecEnv
warnings.filterwarnings('ignore')

WORKING = Path('/kaggle/working')
MODELS = WORKING / 'modelos_onnx'
MODELS.mkdir(exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cudnn.benchmark = True
else:
    print('WARNING: No hay GPU. El entrenamiento sera muy lento.')

In [ ]:
# Celda 2 — Cargar CSVs desde tu dataset 'quantumhive_fusion'
INPUT = Path('/kaggle/input/quantumhive_fusion')
print('Archivos en dataset:')
for f in sorted(INPUT.glob('*')):
    mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:20s} {mb:8.1f} MB')

In [ ]:
# Celda 3 — Leer CSVs de MT5 (tab-separated, headers <DATE> etc)
def leer_csv(nombre: str) -> pd.DataFrame | None:
    ruta = INPUT / nombre
    if not ruta.exists():
        print(f'[SKIP] No existe: {nombre}')
        return None
    df = pd.read_csv(ruta, sep='\t', engine='python')
    df.columns = [c.strip().strip('<>') for c in df.columns]
    df['datetime'] = pd.to_datetime(df['DATE'] + ' ' + df['TIME'],
                                    format='%Y.%m.%d %H:%M:%S', errors='coerce')
    for col in ['OPEN','HIGH','LOW','CLOSE','TICKVOL','VOL','SPREAD']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',','.'), errors='coerce')
    return df.dropna(subset=['datetime','CLOSE']).set_index('datetime').sort_index()

m1 = leer_csv('datatb.csv')
tf_data = {
    'M5': leer_csv('datatb.m5.csv'),
    'M15': leer_csv('datatb.m15.csv'),
    'H1': leer_csv('datatb.H1.csv'),
    'H4': leer_csv('datatb.H4.csv'),
    'D1': leer_csv('datatb.daily.csv'),
}
print(f'\nM1: {len(m1):,} filas')
for k,v in tf_data.items():
    print(f'{k}: {len(v):,} filas' if v is not None else f'{k}: None')

In [ ]:
# Celda 4 — Indicadores tecnicos en M1 + BB% + BBW
def calcular_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    close, high, low = d['CLOSE'], d['HIGH'], d['LOW']
    vol = d.get('TICKVOL', d.get('VOL', pd.Series(1, index=d.index)))

    # RSI 14
    delta = close.diff()
    gain = delta.where(delta>0,0).rolling(14).mean()
    loss = (-delta.where(delta<0,0)).rolling(14).mean()
    rs = gain/loss
    d['rsi'] = 100 - (100/(1+rs))

    # EMAs
    d['ema_fast'] = close.ewm(span=12, adjust=False).mean()
    d['ema_slow'] = close.ewm(span=26, adjust=False).mean()
    d['ema_200'] = close.ewm(span=200, adjust=False).mean()

    # Bollinger + BB% + BBW
    sma20 = close.rolling(20).mean()
    std20 = close.rolling(20).std()
    d['bb_upper'] = sma20 + 2*std20
    d['bb_mid'] = sma20
    d['bb_lower'] = sma20 - 2*std20
    bw = d['bb_upper'] - d['bb_lower']
    d['bb_pct_b'] = (close - d['bb_lower']) / bw
    d['bbw'] = bw / d['bb_mid']

    # ATR 14
    tr1 = high - low
    tr2 = abs(high - close.shift())
    tr3 = abs(low - close.shift())
    d['atr'] = pd.concat([tr1,tr2,tr3],axis=1).max(axis=1).rolling(14).mean()

    # ADX (Wilder)
    tr = pd.concat([tr1,tr2,tr3],axis=1).max(axis=1)
    plus_dm = high.diff().where(high.diff()>low.diff(),0)
    minus_dm = (-low.diff()).where(low.diff()>high.diff(),0)
    atr_s = tr.ewm(alpha=1/14,adjust=False).mean()
    pdi = 100*plus_dm.ewm(alpha=1/14,adjust=False).mean()/atr_s
    mdi = 100*minus_dm.ewm(alpha=1/14,adjust=False).mean()/atr_s
    dx = 100*abs(pdi-mdi)/(pdi+mdi)
    d['adx'] = dx.ewm(alpha=1/14,adjust=False).mean()
    d['plus_di'], d['minus_di'] = pdi, mdi

    # MACD
    ema12 = close.ewm(span=12,adjust=False).mean()
    ema26 = close.ewm(span=26,adjust=False).mean()
    d['macd'] = ema12 - ema26
    d['macd_signal'] = d['macd'].ewm(span=9,adjust=False).mean()
    d['macd_hist'] = d['macd'] - d['macd_signal']

    # Volumen
    d['volume_spike'] = vol / vol.ewm(span=20,adjust=False).mean()
    return d

m1 = calcular_indicadores(m1)
print(f'Indicadores OK. NaN global: {m1.isna().mean().mean()*100:.2f}%')

In [ ]:
# Celda 5 — Alinear timeframes superiores a M1 (merge_asof)
df = m1.reset_index()
for tf_name, tf_df in tf_data.items():
    if tf_df is None or tf_df.empty:
        continue
    cols = tf_df[['OPEN','HIGH','LOW','CLOSE','SPREAD']].copy()
    cols.columns = [f'{c.lower()}_{tf_name.lower()}' for c in cols.columns]
    cols = cols.reset_index()
    df = pd.merge_asof(
        df.sort_values('datetime'),
        cols.sort_values('datetime'),
        on='datetime',
        direction='backward'
    )
df = df.set_index('datetime').sort_index()
print(f'Fusion OK: {len(df):,} filas x {len(df.columns)} columnas')

In [ ]:
# Celda 6 — Confluencias multitemporales + Targets para RL
# EMA trend por TF
for tf in ['m1','m5','m15','h1','h4','daily']:
    col = f'close_{tf}'
    if col in df.columns:
        ema = df[col].ewm(span=20,adjust=False).mean()
        df[f'ema_trend_{tf}'] = np.where(df[col]>ema*1.0002, 1,
                                         np.where(df[col]<ema*0.9998,-1,0))

# RSI signal
df['rsi_signal'] = np.where(df['rsi']>70, -1, np.where(df['rsi']<30, 1, 0))

# BB position
df['bb_position'] = np.where(df['CLOSE']<=df['bb_lower']*1.0001, -1,
                              np.where(df['CLOSE']>=df['bb_upper']*0.9999, 1, 0))

# MACD confluence
df['macd_confluence'] = np.where(
    (df['macd']>df['macd_signal'])&(df['macd_hist']>df['macd_hist'].shift(1)),1,
    np.where((df['macd']<df['macd_signal'])&(df['macd_hist']<df['macd_hist'].shift(1)),-1,0))

# DI confluence
df['di_confluence'] = np.where(df['plus_di']>df['minus_di'],1,
                               np.where(df['plus_di']<df['minus_di'],-1,0))

# Targets (horizon 60 min)
future = df['CLOSE'].shift(-60)
df['return_future'] = (future - df['CLOSE'])/df['CLOSE']
thr = 0.001
df['target_direction'] = np.where(df['return_future']>thr, 2,
                                   np.where(df['return_future']<-thr, 0, 1))
df['target_reward'] = df['return_future'] / (df['atr']/df['CLOSE'])

# Limpiar warmup EMA200
min_idx = df['ema_200'].first_valid_index()
df = df.loc[min_idx:].dropna(subset=['target_direction'])

print(f'Dataset final: {len(df):,} filas')
print(f'Target LONG={(df.target_direction==2).mean()*100:.1f}% | FLAT={(df.target_direction==1).mean()*100:.1f}% | SHORT={(df.target_direction==0).mean()*100:.1f}%')

In [ ]:
# Celda 7 — Guardar datatb_fusion (parquet es mas rapido)
ruta_parquet = WORKING / 'datatb_fusion.parquet'
df.to_parquet(ruta_parquet, compression='zstd')
mb = ruta_parquet.stat().st_size / 1024 / 1024
print(f'Guardado: {ruta_parquet} ({mb:.1f} MB)')

# Link de descarga
from IPython.display import FileLink
FileLink(ruta_parquet)

---\n## FASE B: Entrenamiento RL (GPU T4)

In [ ]:
# Celda 8 — Entorno RL simplificado para Kaggle
class EntornoKaggle(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, df: pd.DataFrame, features: list[str], horizon: int = 60):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.features = [f for f in features if f in df.columns]
        self.horizon = horizon
        self.window = 200
        self.action_space = gym.spaces.Discrete(3)
        obs_len = len(self.features) + 3
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(obs_len,), dtype=np.float32)
        self._idx = self.window
        self._last_pos = 1
    def reset(self, seed=None, options=None):
        self._idx = self.window
        self._last_pos = 1
        return self._obs(), {}
    def _obs(self):
        row = self.df.iloc[self._idx]
        obs = list(row[self.features].fillna(0).values)
        obs += [self._last_pos, row.get('bb_pct_b',0.5), row.get('bbw',0.05)]
        return np.array(obs, dtype=np.float32)
    def step(self, action):
        row = self.df.iloc[self._idx]
        end = min(self._idx + self.horizon, len(self.df)-1)
        future_price = self.df.iloc[end]['CLOSE']
        ret = (future_price - row['CLOSE']) / row['CLOSE']
        if action == 0:    # SHORT
            reward = -ret * 100 - abs(ret)*0.5
        elif action == 2:  # LONG
            reward = ret * 100 - abs(ret)*0.5
        else:              # WAIT
            reward = -abs(ret)*10
        self._last_pos = action
        self._idx += 1
        terminated = self._idx >= len(self.df) - self.horizon - 1
        return self._obs(), float(reward), terminated, False, {'return': ret}

FEATURES = [
    'CLOSE','HIGH','LOW','OPEN','volume_spike',
    'rsi','ema_fast','ema_slow','bb_upper','bb_lower',
    'atr','adx','macd','macd_signal',
    'ema_trend_m1','ema_trend_m5','ema_trend_h1','macd_confluence','bb_position'
]
print(f'Entorno definido. Features: {len(FEATURES)}')

In [ ]:
# Celda 9 — Helper evaluacion
def evaluar(model, env, n_steps=3000):
    obs, _ = env.reset()
    rewards, rets, wins, losses = [], [], 0, 0
    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, trunc, info = env.step(int(action))
        rewards.append(reward)
        ret = info.get('return',0)
        rets.append(ret)
        if ret > 0.001: wins += 1
        elif ret < -0.001: losses += 1
        if done or trunc: break
    n = wins+losses
    return {
        'mean_reward': float(np.mean(rewards)),
        'winrate': round(wins/n, 4) if n>0 else 0,
        'profit_factor': round(abs(sum(r for r in rets if r>0))/abs(sum(r for r in rets if r<0)), 2)
                if any(r<0 for r in rets) else 999,
        'trades': n,
        'total_return_pct': round(sum(rets)*100, 3)
    }
print('Eval helper OK')

In [ ]:
# Celda 10 — BOT MADRE (RecurrentPPO)
print('=== BOT MADRE ===')
env_madre = DummyVecEnv([lambda: EntornoKaggle(df, FEATURES)])

model_madre = RecurrentPPO(
    'MlpLstmPolicy',
    env_madre,
    verbose=1,
    device=DEVICE,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.995,
    gae_lambda=0.95
)
model_madre.learn(total_timesteps=300_000)

stats_madre = evaluar(model_madre, EntornoKaggle(df, FEATURES), n_steps=5000)
print(f'MADRE: {stats_madre}')
model_madre.save(str(MODELS / 'bot_madre_ppo'))

In [ ]:
# Celda 11 — BOT REVERSION (filtra RSI extremos)
print('\n=== BOT REVERSION ===')
df_rev = df[(df['rsi']>70) | (df['rsi']<30)].copy()
env_rev = DummyVecEnv([lambda: EntornoKaggle(df_rev, FEATURES)])

model_rev = PPO('MlpPolicy', env_rev, verbose=1, device=DEVICE,
                learning_rate=3e-4, n_steps=1024, batch_size=128, n_epochs=10)
model_rev.learn(total_timesteps=200_000)

stats_rev = evaluar(model_rev, EntornoKaggle(df_rev, FEATURES), n_steps=3000)
print(f'REVERSION: {stats_rev}')
model_rev.save(str(MODELS / 'bot_reversion_ppo'))

In [ ]:
# Celda 12 — BOT CONTINUACION (filtra ADX>25)
print('\n=== BOT CONTINUACION ===')
df_cont = df[df['adx'] > 25].copy()
env_cont = DummyVecEnv([lambda: EntornoKaggle(df_cont, FEATURES)])

model_cont = PPO('MlpPolicy', env_cont, verbose=1, device=DEVICE,
                 learning_rate=3e-4, n_steps=1024, batch_size=128, n_epochs=10)
model_cont.learn(total_timesteps=200_000)

stats_cont = evaluar(model_cont, EntornoKaggle(df_cont, FEATURES), n_steps=3000)
print(f'CONTINUACION: {stats_cont}')
model_cont.save(str(MODELS / 'bot_continuacion_ppo'))

In [ ]:
# Celda 13 — BOT SCALPER (filtra BB% extremos)
print('\n=== BOT SCALPER ===')
df_scalp = df[(df['bb_pct_b']<0.05) | (df['bb_pct_b']>0.95)].copy()
env_scalp = DummyVecEnv([lambda: EntornoKaggle(df_scalp, FEATURES)])

model_scalp = PPO('MlpPolicy', env_scalp, verbose=1, device=DEVICE,
                  learning_rate=3e-4, n_steps=1024, batch_size=128, n_epochs=10)
model_scalp.learn(total_timesteps=200_000)

stats_scalp = evaluar(model_scalp, EntornoKaggle(df_scalp, FEATURES), n_steps=3000)
print(f'SCALPER: {stats_scalp}')
model_scalp.save(str(MODELS / 'bot_scalper_ppo'))

---\n## FASE C: Exportar ONNX para MT5

In [ ]:
# Celda 14 — Exportar ONNX (opset 12)
import onnx

dummy_env = EntornoKaggle(df.head(1000), FEATURES)
obs_sample, _ = dummy_env.reset()
obs_t = torch.as_tensor(obs_sample, dtype=torch.float32).unsqueeze(0)
if DEVICE == 'cuda':
    obs_t = obs_t.cuda()

for nombre, modelo in [('madre',model_madre), ('reversion',model_rev),
                       ('continuacion',model_cont), ('scalper',model_scalp)]:
    path = str(MODELS / f'bot_{nombre}.onnx')
    try:
        # Extraer actor MLP (sin LSTM por compatibilidad MT5)
        if hasattr(modelo.policy, 'mlp_extractor'):
            actor = modelo.policy.mlp_extractor
        else:
            actor = modelo.policy
        torch.onnx.export(
            actor, obs_t, path,
            export_params=True, opset_version=12,
            input_names=['obs'],
            output_names=['features'],
            dynamic_axes={'obs': {0: 'batch'}, 'features': {0: 'batch'}}
        )
        m = onnx.load(path)
        onnx.checker.check_model(m)
        kb = Path(path).stat().st_size / 1024
        print(f'ok bot_{nombre}.onnx ({kb:.0f} KB)')
    except Exception as e:
        print(f'ERROR {nombre}: {e}')

In [ ]:
# Celda 15 — Reporte final + links de descarga
reporte = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'device': DEVICE,
    'filas_dataset': len(df),
    'bots': {
        'madre': stats_madre,
        'reversion': stats_rev,
        'continuacion': stats_cont,
        'scalper': stats_scalp
    },
    'archivos': [f.name for f in MODELS.iterdir()]
}

with open(WORKING / 'reporte_kaggle.json', 'w') as f:
    json.dump(reporte, f, indent=2)

print('\n' + '='*60)
print('RESULTADOS FINALES')
print('='*60)
for n,s in reporte['bots'].items():
    wr = s.get('winrate',0)
    pf = s.get('profit_factor',0)
    mr = s.get('mean_reward',0)
    print(f'{n:12s} | WR={wr:.2%} | PF={pf} | MR={mr:.4f}')
print('='*60)

# Empaquetar todo en ZIP para descargar
import shutil
zip_path = WORKING / 'modelos_onnx.zip'
shutil.make_archive(str(WORKING / 'modelos_onnx'), 'zip', str(MODELS))
print(f'\nZIP listo: {zip_path}')

print('\nLinks de descarga:')
for f in [WORKING/'modelos_onnx.zip', WORKING/'datatb_fusion.parquet', WORKING/'reporte_kaggle.json']:
    if f.exists():
        display(FileLink(f))